In [1]:
import pandas as pd

# Directly read your JSONL file
df = pd.read_json(r"C:\Users\naray\OneDrive\Desktop\internship 3\task_2\quotes.jsonl", lines=True)
# ✅ STEP 1: Preprocess quotes
def preprocess_quotes(df):
    clean_data = []
    for _, row in df.iterrows():
        quote = str(row.get('quote', '')).strip().lower()
        author = str(row.get('author', 'unknown')).strip().lower()
        tags = [tag.strip().lower() for tag in row.get('tags', []) if tag.strip()]

        if quote:
            clean_data.append({
                'quote': quote,
                'author': author,
                'tags': tags
            })
    return clean_data

quotes_data = preprocess_quotes(df)

# Preview
print(df.head())


                                               quote                 author  \
0     “Be yourself; everyone else is already taken.”            Oscar Wilde   
1  “I'm selfish, impatient and a little insecure....         Marilyn Monroe   
2  “Two things are infinite: the universe and hum...        Albert Einstein   
3                   “So many books, so little time.”            Frank Zappa   
4  “A room without books is like a body without a...  Marcus Tullius Cicero   

                                                tags  
0  [be-yourself, gilbert-perreira, honesty, inspi...  
1  [best, life, love, mistakes, out-of-control, t...  
2  [human-nature, humor, infinity, philosophy, sc...  
3                                     [books, humor]  
4                              [books, simile, soul]  


In [2]:
df.to_csv("quotes_dataset.csv", index=False)


In [3]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Prepare training samples (quote vs. enriched text)
train_samples = []
for item in quotes_data:
    meta_text = f"{item['author']} - {', '.join(item['tags'])}"
    train_samples.append(InputExample(texts=[item['quote'], meta_text]))


# Load base model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Patch to disable model card generation
class DummyModelCardData:
    def __init__(self):
        self.train_datasets = []
        self.eval_datasets = []
        self.widget = False  # Prevent widget logic
    def add_tags(self, *args, **kwargs): pass
    def extract_dataset_metadata(self, *args, **kwargs): return []
    def set_losses(self, losses): pass
    def set_widget_examples(self, examples): pass



model.model_card_data = DummyModelCardData()
# DataLoader & training setup
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    show_progress_bar=True
)



C:\Users\naray\anaconda3\envs\rag-quotes\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [4]:
model.save("fine-tuned-quote-model", create_model_card=False)


In [6]:
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, util

# Load your fine-tuned model
model = SentenceTransformer("fine-tuned-quote-model")

# Your quotes_data loaded from df (list of dicts with 'quote', 'author', 'tags')
# Example:
# quotes_data = [
#     {"quote": "...", "author": "...", "tags": [...]},
#     ...
# ]

# Encode all quotes (only once, outside functions)
all_quotes = [q['quote'] for q in quotes_data]
quote_embeddings = model.encode(all_quotes, convert_to_tensor=True)
embeddings_np = quote_embeddings.cpu().detach().numpy().astype('float32')

# Build FAISS index
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

# Parse tags and author from query
def parse_advanced_query(query):
    tags = []
    author = None
    tag_match = re.search(r"tagged with both ['“”‘’\"]?(\w+)['“”‘’\"]?\s+and\s+['“”‘’\"]?(\w+)", query, re.IGNORECASE)
    if tag_match:
        tags = [tag_match.group(1).lower(), tag_match.group(2).lower()]
    else:
        tag_match = re.search(r"tagged ['“”‘’\"]?(\w+)['“”‘’\"]?", query, re.IGNORECASE)
        if tag_match:
            tags = [tag_match.group(1).lower()]

    author_match = re.search(r"by ([\w\s.]+)", query, re.IGNORECASE)
    if author_match:
        author = author_match.group(1).strip().lower()
    return tags, author

def retrieve_quotes(query, top_k=3):
    filter_tags, filter_author = parse_advanced_query(query)

    # Filter quotes by tag/author
    filtered_quotes = []
    filtered_embeddings = []
    for i, q in enumerate(quotes_data):
        q_tags = [t.lower() for t in q['tags']]
        q_author = q['author'].lower()

        tag_match = all(tag in q_tags for tag in filter_tags) if filter_tags else True
        author_match = (filter_author in q_author) if filter_author else True

        if tag_match and author_match:
            filtered_quotes.append(q)
            filtered_embeddings.append(embeddings_np[i])

    if not filtered_quotes:
        return []

    # Clean query to remove filters for embedding
    clean_query = re.sub(r"tagged with.*|tagged.*|by .*", "", query, flags=re.IGNORECASE).strip()
    query_embedding = model.encode(clean_query, convert_to_tensor=True)
    query_np = query_embedding.cpu().detach().numpy().astype('float32').reshape(1, -1)

    # Search top-k in filtered quotes
    temp_index = faiss.IndexFlatL2(query_np.shape[1])
    temp_index.add(np.array(filtered_embeddings).astype('float32'))
    distances, indices = temp_index.search(query_np, min(top_k, len(filtered_quotes)))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        q = filtered_quotes[idx]
        results.append({
            'quote': q['quote'],
            'author': q['author'],
            'tags': q['tags'],
            'distance': dist
        })
    return results

# Your Huggingface function stub (replace with your actual function)
def generate_answer_with_huggingface(query, context_quotes, hf_token):
    # Example placeholder:
    return f"Answer generated for '{query}' based on {len(context_quotes)} quotes."


query = "quotes about love and bible by C.S. Lewis"

context_quotes = retrieve_quotes(query, top_k=3)

if not context_quotes:
    print("No relevant quotes found.")
else:
    generated_answer = generate_answer_with_huggingface(query, context_quotes, hf_token)
    print("Generated Answer:", generated_answer)


Generated Answer: Answer generated for 'quotes about love and bible by C.S. Lewis' based on 3 quotes.


In [7]:
if not context_quotes:
    print("No relevant quotes found.")
else:
    print(f"Retrieved {len(context_quotes)} quotes:")
    for i, q in enumerate(context_quotes, 1):
        print(f"Quote {i}: \"{q['quote']}\" - {q['author']} [Tags: {q['tags']}]")
    generated_answer = generate_answer_with_huggingface(query, context_quotes, hf_token)
    print("\nGenerated Answer:", generated_answer)


Retrieved 3 quotes:
Quote 1: "“the christian does not think god will love us because we are good, but that god will make us good because he loves us.”" - c.s. lewis [Tags: ['christianity', 'god', 'religion']]
Quote 2: "“love is not affectionate feeling, but a steady wish for the loved person's ultimate good as far as it can be obtained.”" - c.s. lewis [Tags: ['love']]
Quote 3: "“god can't give us peace and happiness apart from himself because there is no such thing.”" - c.s. lewis [Tags: ['god-religion-happiness']]

Generated Answer: Answer generated for 'quotes about love and bible by C.S. Lewis' based on 3 quotes.


In [ ]:
st.set_page_config(page_title="Quote Retriever", page_icon="💬")
st.title("💬 Semantic Quote Retriever")

query = st.text_input("Enter your query", placeholder="e.g. Motivational quotes tagged 'accomplishment'")
top_k = st.slider("Number of top quotes", 1, 10, 3)

if st.button("Retrieve Quotes"):
    if not query.strip():
        st.warning("Please enter a query.")
    else:
        with st.spinner("Searching..."):
            results = retrieve_quotes(query, top_k=top_k)

        if not results:
            st.error("No relevant quotes found.")
        else:
            st.subheader("📜 Retrieved Quotes")
            for res in results:
                st.markdown(f"> {res['quote']}  \n— *{res['author']}*")
                st.caption(f"Tags: {', '.join(res['tags'])} | Distance: {res['distance']:.4f}")

            answer = generate_answer_with_huggingface(query, results)
            st.subheader("🧠 Generated Answer")
            st.success(answer)